In [1]:
import os
import sys

# Standard interactive replacement for the 'parent directory' hack
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

from PPRCalculator import PPRCalculator
from ModelData import ModelData

In [2]:
# All real models where model number > 10000 (model number = 2nd token in filename)
ewe_dir = Path('../real_models/new_EwE_jsons')
models_over_10000 = [
    f.name for f in sorted(ewe_dir.glob('*.json'))
    if ModelData._parse_filename(f.stem)[0] > 10000
]
for i, name in enumerate(models_over_10000):
    print(i, name)

0 13_10013_Humboldt_Current_(1980).json
1 13_11013_Humboldt_Current_(1995-2004).json
2 36_10036_Northern_South_China_Sea_(1970s).json
3 36_11036_Northern_South_China_Sea_(2000s).json
4 36_12036_Northern_South_China_Sea_Ecobase_(1970).json
5 47_10047_East_China_Sea_(1997).json
6 47_11047_East_China_Sea_(2018).json
7 48_10048_North_Yellow_Sea_(2019).json
8 48_11048_North_Yellow_Sea_(2019).json
9 48_12048_Southwestern_Yellow_Sea_(2008).json
10 49_10049_Kuroshio_Current_(2013).json


### ecopath model comparison

In [21]:
model_path_new = f'../real_models/new_EwE_jsons/{models_over_10000[4]}'
model_path_orig = f'../real_models/EwE_jsons/410_410_North_South_of_China_Sea_(1970).json'

model_new = ModelData(model_path_new)
model_orig = ModelData(model_path_orig)
self_new = PPRCalculator.from_modeldata(model_new, underdetermined=True, zero_biomass_accum=False)
self_orig = PPRCalculator.from_modeldata(model_orig, underdetermined=True, zero_biomass_accum=False)

df_new = self_new.get_groups_df()
df_orig = self_orig.get_groups_df()
df_orig.columns

Index(['group_name', 'trophic_info', 'taxon_descr', 'tl', 'ge', 'ee', 'catch',
       'biomass', 'pb', 'qb', 'p', 'q', 'predation', 'M0', 'gs', 'egestion',
       'respiration', 'biomass_accum', 'emigration', 'immigration',
       'net_migration', 'flow_to_det', 'detritus_import'],
      dtype='object')

In [22]:
# self = self_orig
self = self_new

In [11]:
col = 'biomass_accum'
bool(np.all(np.isclose(df_orig[col] - df_new[col], 0)))
# df_new[col]

True

In [13]:
# model_path = '../real_models/EwE_jsons/227_227_Iceland_(1950).json'
# model_path = '../real_models/ToyModels/900_900_Multi_DET_Toy_(2026).json'
# model_path = '../real_models/EwE_jsons/68_68_Icelandic_shelf_(1997).json'
model_path = '../real_models/EwE_jsons/410_410_North_South_of_China_Sea_(1970).json'

# model_path = f'../real_models/new_EwE_jsons/{models_over_10000[4]}'

model = ModelData(model_path)
self = PPRCalculator.from_modeldata(model, underdetermined=True, zero_biomass_accum=False)
# self = PPRCalculator.from_modeldata(model, underdetermined=False, zero_biomass_accum=False)
self.get_groups_df()

,group_name,trophic_info,taxon_descr,tl,ge,ee,catch,biomass,pb,qb,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
39,diet_import,Import,NaN,1.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
38,Detritus,DET,None,NaN,1.000000,1.000000,0.000000,100.000000,1291.868683,1291.868683,...,0.000000,0.0,0.000000,0.000000,127035.389678,0.0,0.0,0.0,0.000000,0.0
37,Seaturtles,Regular,None,NaN,0.028571,0.503045,0.000010,0.000200,0.100000,3.500000,...,0.000010,0.2,0.000140,0.000540,0.000000,0.0,0.0,0.0,0.000150,0.0
36,Other mammals,Regular,None,NaN,0.010643,0.068125,0.000120,0.015800,0.112000,10.523000,...,0.001649,0.2,0.033253,0.131241,0.000000,0.0,0.0,0.0,0.034902,0.0
35,Pinnipeds,Regular,None,NaN,0.003047,0.679270,0.000140,0.004600,0.045000,14.768000,...,0.000066,0.2,0.013587,0.054139,0.000000,0.0,0.0,0.0,0.013653,0.0
34,Seabirds,Regular,None,NaN,0.000885,0.004618,0.000000,0.002198,0.060000,67.759000,...,0.000131,0.2,0.029787,0.119016,0.000000,0.0,0.0,0.0,0.029918,0.0
33,Pelagic sharks and rays,Regular,None,NaN,0.200000,0.500000,0.005042,0.028412,0.390000,1.950000,...,0.005540,0.2,0.011081,0.033242,0.000000,0.0,0.0,0.0,0.016621,0.0
32,Demersal sharks and rays,Regular,None,NaN,0.200000,0.364069,0.015831,0.040000,1.260000,6.300000,...,0.032051,0.2,0.050400,0.151200,0.000000,0.0,0.0,0.0,0.082451,0.0
31,Pelagic fish (30+cm),Regular,None,NaN,0.143381,0.282512,0.038000,0.158000,0.900000,6.276998,...,0.102027,0.2,0.198353,0.651213,0.000000,0.0,0.0,0.0,0.300380,0.0


In [23]:
balanced, p, q = self.is_model_balanced()
print(f'balanced: {balanced}')
groups_df = self.get_groups_df()
groups_df['p-p`'] = p - groups_df['p']
groups_df['q-q`'] = q - groups_df['q']
groups_df.round(5)

balanced: False


,group_name,trophic_info,taxon_descr,tl,ge,ee,catch,biomass,pb,qb,...,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import,p-p`,q-q`
group_seq,,,,,,,,,,,,,,,,,,,,,
39,diet_import,Import,NaN,1.0,0.00000,0.00000,0.00000,1.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.0,0.0,0.0,0.00000,0.0,0.00000,0.00000
38,Detritus,DET,None,NaN,1.00000,1.00000,0.00000,100.00000,1291.86868,1291.86868,...,0.00000,0.00000,127035.38968,0.0,0.0,0.0,0.00000,0.0,0.00000,0.00000
37,Seaturtles,Regular,None,NaN,0.02857,0.50304,0.00001,0.00020,0.10000,3.50000,...,0.00014,0.00054,0.00000,0.0,0.0,0.0,0.00015,0.0,0.00000,0.00000
36,Other mammals,Regular,None,NaN,0.01064,0.06812,0.00012,0.01580,0.11200,10.52300,...,0.03325,0.13124,0.00000,0.0,0.0,0.0,0.03490,0.0,0.00000,-0.00000
35,Pinnipeds,Regular,None,NaN,0.00305,0.67927,0.00014,0.00460,0.04500,14.76800,...,0.01359,0.05414,0.00000,0.0,0.0,0.0,0.01365,0.0,0.00000,-0.00000
34,Seabirds,Regular,None,NaN,0.00089,0.00462,0.00000,0.00220,0.06000,67.75900,...,0.02979,0.11902,0.00000,0.0,0.0,0.0,0.02992,0.0,-0.00000,0.00000
33,Pelagic sharks and rays,Regular,None,NaN,0.20000,0.50000,0.00504,0.02841,0.39000,1.95000,...,0.01108,0.03324,0.00000,0.0,0.0,0.0,0.01662,0.0,-0.00000,-0.00000
32,Demersal sharks and rays,Regular,None,NaN,0.20000,0.36407,0.01583,0.04000,1.26000,6.30000,...,0.05040,0.15120,0.00000,0.0,0.0,0.0,0.08245,0.0,-0.00000,-0.00000
31,Pelagic fish (30+cm),Regular,None,NaN,0.14338,0.28251,0.03800,0.15800,0.90000,6.27700,...,0.19835,0.65121,0.00000,0.0,0.0,0.0,0.30038,0.0,-0.00000,0.00000


### model-specific tests

In [3]:
model_path = f'../real_models/new_EwE_jsons/{models_over_10000[5]}'

model = ModelData(model_path)
self = PPRCalculator.from_modeldata(model, underdetermined=True, zero_biomass_accum=False)

self.get_groups_df()

,group_name,trophic_info,taxon_descr,tl,ge,ee,catch,biomass,pb,qb,...,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import,det_export
group_seq,,,,,,,,,,,,,,,,,,,,,
25,diet_import,Import,NaN,1.0,0.000000,0.000,0.0,1.00000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
24,Detritus,DET,-,NaN,1.000000,1.000,0.0,100.00000,24.119104,24.119104,...,0.0,0.0,0.000000,2025.626983,0.0,0.0,0.0,0.000000,0.0,0.0
23,Marine mammals,Regular,Sousa chinensis,NaN,0.001667,0.000,0.0,0.00404,0.050000,30.000000,...,0.0,0.0,0.120998,0.000000,0.0,0.0,0.0,0.000202,0.0,0.0
22,Sharks,Regular,Scoliodon laticaudus,NaN,0.156250,0.000,0.0,0.00889,0.500000,3.200000,...,0.0,0.0,0.024003,0.000000,0.0,0.0,0.0,0.004445,0.0,0.0
21,Small yellow croakers,Regular,Larimichthys polyactis,NaN,0.477937,0.226,0.0,0.35600,4.300000,8.997000,...,0.0,0.0,1.672132,0.124464,0.0,0.0,0.0,1.184839,0.0,0.0
20,Large yellow croakers,Regular,Larimichthys crocea,NaN,0.433544,0.997,0.0,0.00107,2.130000,4.913000,...,0.0,0.0,0.002978,0.002158,0.0,0.0,0.0,0.000007,0.0,0.0
19,Omnivores,Regular,"Ilisha elongate, Nibea albiflora, Ostorhinchus...",NaN,0.410285,0.999,0.0,0.13600,3.279000,7.992000,...,0.0,0.0,0.640968,0.408925,0.0,0.0,0.0,0.000446,0.0,0.0
18,Benthivores/Piscivores,Regular,"Pennahia argentatus, Nemipterus virgatus, Piso...",NaN,0.419213,0.985,0.0,0.53400,3.910000,9.327000,...,0.0,0.0,2.892678,2.022235,0.0,0.0,0.0,0.031319,0.0,0.0
17,Planktivores/Piscivores,Regular,"Setipinna tenuifilis, Psenopsis anomala, Scomb...",NaN,0.075512,0.353,0.0,2.58800,0.885000,11.720000,...,0.0,0.0,28.040980,0.253034,0.0,0.0,0.0,1.481876,0.0,0.0


In [4]:
sppr, sppr_det = self._sample_SPPR_new_forced_balance(TE_option='TE')
sppr

c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\PPRCalculator.py:1944: RuntimeWarning: 2 group(s) have TE=0 because EE=0 (M0=p): 23 (Marine mammals), 22 (Sharks). Their consumed PP is re-credited to detritus (fix_EE_0_cases=True) so the TE SPPR stays PP-balanced.
  warnings.warn(


,25,24,1
25,1.0,0.000000,0.000000
24,0.0,0.966130,0.000000
1,0.0,0.000000,1.000000
23,0.0,0.000000,0.000000
22,0.0,0.000000,0.000000
21,0.0,122.916462,199.984041
20,0.0,20.296241,32.602964
19,0.0,30.596284,45.483756
18,0.0,16.485812,20.178818
17,0.0,297.635263,396.919939


In [9]:
sppr, A, L = self.SPPR_new(TE=None, TE_option='TE', collapse_det=False, fix_EE_0_cases=True)
sppr, A, L = PPRCalculator.rename_results([sppr, A, L], self.seq2name)
print(self.is_model_balanced()[0])
print(self.get_PPR(sppr, only_inner=True, only_pp=False).sum(axis=1))
print(self.get_PPR(sppr, only_inner=True, only_pp=False).rename(columns=self.seq2name))
print(f'balanced: {self.is_sppr_balanced(sppr)}')
sppr.round(5)

True
catch    0.0
dtype: float64
       Phytoplankton  Detritus
catch            0.0       0.0
balanced: (True, np.float64(2819.964), np.float64(2819.964000000008))


c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\PPRCalculator.py:1944: RuntimeWarning: 2 group(s) have TE=0 because EE=0 (M0=p): 23 (Marine mammals), 22 (Sharks). Their consumed PP is re-credited to detritus (fix_EE_0_cases=True) so the TE SPPR stays PP-balanced.
  warnings.warn(


,diet_import,Detritus,Phytoplankton
diet_import,1.0,0.00000,0.00000
Detritus,0.0,0.96613,0.00000
Marine mammals,0.0,0.00000,0.00000
Sharks,0.0,0.00000,0.00000
Small yellow croakers,0.0,122.91646,199.98404
Large yellow croakers,0.0,20.29624,32.60296
Omnivores,0.0,30.59628,45.48376
Benthivores/Piscivores,0.0,16.48581,20.17882
Planktivores/Piscivores,0.0,297.63526,396.91994
Planktivores/Benthivores,0.0,36.15458,86.86277


In [11]:
sppr, A, L = self.SPPR_2015(only_pp_det=True)
sppr, A, L = PPRCalculator.rename_results([sppr, A, L], self.seq2name)
print(self.is_model_balanced()[0])
print(self.get_PPR(sppr, only_inner=True, only_pp=False).sum(axis=1))
print(self.get_PPR(sppr, only_inner=True, only_pp=False).rename(columns=self.seq2name))
print(f'balanced: {self.is_sppr_balanced(sppr)}')
sppr.round(5)

True
catch    0.0
dtype: float64
       Phytoplankton
catch            0.0
balanced: (False, np.float64(2819.964), np.float64(2733.0956086624856))


,diet_import,Phytoplankton
group_seq,,
diet_import,1.0,0.00000
Detritus,0.0,0.92950
Marine mammals,1.0,0.00000
Sharks,1.0,0.00000
Small yellow croakers,0.0,318.24026
Large yellow croakers,0.0,52.12970
Omnivores,0.0,74.92002
Benthivores/Piscivores,0.0,36.03959
Planktivores/Piscivores,0.0,683.27069


In [25]:
# sppr, A, L = self.SPPR_EwE(TE_option='TE', silent=False)
# sppr, A, L = PPRCalculator.rename_results([sppr, A, L], self.seq2name)
# print(self.is_model_balanced()[0])
# print(self.get_PPR(sppr, only_inner=True).sum(axis=1))
# print(f'balanced: {self.is_sppr_balanced(sppr)}')
# sppr.round(3)

In [29]:
# sppr = self.SPPR_1995(global_TE=0.1)
# sppr = self.SPPR_1995(global_TE='mean')
# sppr, A, L = self.SPPR_EwE_Ido(TE_option='TE', use_EE=False)
# sppr,  = PPRCalculator.rename_results([sppr], self.seq2name)
# print(self.is_model_balanced()[0])
# print(self.get_PPR(sppr, only_inner=True).sum(axis=1))
# print(f'balanced: {self.is_sppr_balanced(sppr)}')
# sppr.round(3)

### cross-models tests

In [4]:
# Pipeline: build a summary DataFrame over every model in new_EwE_jsons
ewe_dir = Path('../real_models/EwE_jsons')
model_files = sorted(ewe_dir.glob('*.json'))

balanced_models = []
not_balanced = []

for f in tqdm(model_files):
    try:
        model = ModelData(str(f))
        self = PPRCalculator.from_modeldata(model, underdetermined=True, zero_biomass_accum=False)
        balanced = self.is_model_balanced()[0]
        # print(f'{f.name} - balanced: {balanced}')

        if balanced:
            balanced_models.append(f.name)
        else:
            not_balanced.append(f.name)

    except:
        continue

  0%|          | 0/230 [00:00<?, ?it/s]c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\ModelData.py:606: RuntimeWarning: 5 group(s) route less than 100% of their flow_to_det to detritus; the remainder is treated as export out of the system: 37 (Shortfinned squid), 17 (Winter flounder), 16 (Witch flounder), 3 (Grey seals), 1 (Walrus).
  warnings.warn(
  3%|▎         | 6/230 [00:00<00:17, 13.15it/s]c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\ModelData.py:606: RuntimeWarning: 2 group(s) route less than 100% of their flow_to_det to detritus; the remainder is treated as export out of the system: 5 (Sei whales), 4 (Brydes whales).
  warnings.warn(
 13%|█▎        | 30/230 [00:02<00:11, 17.24it/s]c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\ModelData.py:606: RuntimeWarning: 1 group(s) route less than 100% of their flow_to_det to detritus; the remainder is treated as export out of the system: 1 (Engraulidae).
  warnings.wa

In [13]:
# Pipeline: build a summary DataFrame over every model in new_EwE_jsons
ewe_dir = Path('../real_models/new_EwE_jsons')
model_files = sorted(ewe_dir.glob('*.json'))

def safe(fn):
    """Run fn() and return its result, or np.nan if it raises."""
    try:
        return fn()
    except Exception as e:
        return np.nan


rows = []
for f in tqdm(model_files):
    model = ModelData(str(f))
    try:
        self = PPRCalculator.from_modeldata(model, underdetermined=True, zero_biomass_accum=False)
    except:
        continue

    # SPPR_new TE_option='TE' (default collapse) -> also reused for (4)
    sppr_TE = safe(lambda: self.SPPR_new(TE=None, TE_option='TE')[0])
    # SPPR_new TE_option='TE' with collapse_det=True
    sppr_TE_collapse = safe(lambda: self.SPPR_new(TE=None, TE_option='TE', collapse_det=True)[0])
    # SPPR_new TE_option='GE'
    sppr_GE = safe(lambda: self.SPPR_new(TE=None, TE_option='GE')[0])
    sppr_TE_collapse = safe(lambda: self.SPPR_new(TE=None, TE_option='GE', collapse_det=True)[0])

    rows.append({
        'filename': f.name,                                                                        # (1)
        'n_detritus': len(self.get_DET_seq()),                                                     # (2)
        'model_balanced': self.is_model_balanced()[0],                                             # (3)
        'sppr_balanced_TE': safe(lambda: self.is_sppr_balanced(sppr_TE)[0]),                       # (4)
        'sppr_balanced_collapse_det_TE': safe(lambda: self.is_sppr_balanced(sppr_TE_collapse)[0]), # (5)
        'sppr_balanced_GE': safe(lambda: self.is_sppr_balanced(sppr_GE)[0]),                       # (6)
        'sppr_balanced_collapse_det_GE': safe(lambda: self.is_sppr_balanced(sppr_TE_collapse)[0]), # (7)
        'PPR2NPP_TE_only_pp': safe(lambda: self.get_PPR2NPP_ratio(sppr_TE, only_pp=True)),         # (8)
        'PPR2NPP_TE_all': safe(lambda: self.get_PPR2NPP_ratio(sppr_TE, only_pp=False)),            # (9)
        'PPR2NPP_GE_only_pp': safe(lambda: self.get_PPR2NPP_ratio(sppr_GE, only_pp=True)),         # (10)
        'PPR2NPP_GE_all': safe(lambda: self.get_PPR2NPP_ratio(sppr_GE, only_pp=False)),            # (11)
        'PPR2NPP_1995_TE0.1': safe(lambda: self.get_PPR2NPP_ratio(self.SPPR_1995(global_TE=0.1))),    # (12)
        'PPR2NPP_1995_TEmean': safe(lambda: self.get_PPR2NPP_ratio(self.SPPR_1995(global_TE='mean'))),# (13)
    })

results_df = pd.DataFrame(rows)
results_df

  0%|          | 0/11 [00:00<?, ?it/s]c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\PPRCalculator.py:1944: RuntimeWarning: 1 group(s) have TE=0 because EE=0 (M0=p): 14 (Sea lions). Their consumed PP is re-credited to detritus (fix_EE_0_cases=True) so the TE SPPR stays PP-balanced.
  warnings.warn(
c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\PPRCalculator.py:1944: RuntimeWarning: 1 group(s) have TE=0 because EE=0 (M0=p): 14 (Sea lions). Their consumed PP is re-credited to detritus (fix_EE_0_cases=True) so the TE SPPR stays PP-balanced.
  warnings.warn(
  9%|▉         | 1/11 [00:00<00:01,  7.47it/s]c:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\FishEstimationAI\PPRCalculator.py:1953: RuntimeWarning: 4 group(s) have TE=0 because EE=0 (M0=p): 33 (Cetaceans), 32 (Pinnipeds), 31 (Seabirds), 7 (Chrysaora plocamia). Their consumed PP is dropped, so the TE SPPR will not balance (multi-DET: re-credit not applied).
  warnings.warn(
c:\Users\

,filename,n_detritus,model_balanced,sppr_balanced_TE,sppr_balanced_collapse_det_TE,sppr_balanced_GE,sppr_balanced_collapse_det_GE,PPR2NPP_TE_only_pp,PPR2NPP_TE_all,PPR2NPP_GE_only_pp,PPR2NPP_GE_all,PPR2NPP_1995_TE0.1,PPR2NPP_1995_TEmean
0,13_10013_Humboldt_Current_(1980).json,1,True,True,True,True,True,0.037599,0.052465,0.007102,0.007567,0.053072,0.022330
1,13_11013_Humboldt_Current_(1995-2004).json,4,True,False,True,True,True,0.042612,0.042612,0.008036,-0.015321,0.018804,0.024769
2,36_10036_Northern_South_China_Sea_(1970s).json,1,True,True,True,True,True,0.036368,0.051911,0.000232,0.000522,0.001316,0.000995
3,36_11036_Northern_South_China_Sea_(2000s).json,1,True,True,True,True,True,0.006871,0.010442,0.000769,0.001428,0.004236,0.001090
4,36_12036_Northern_South_China_Sea_Ecobase_(197...,1,False,False,False,False,False,0.035161,0.051230,0.000282,0.000542,0.001560,0.000711
5,47_10047_East_China_Sea_(1997).json,1,True,True,True,True,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,47_11047_East_China_Sea_(2018).json,1,True,True,True,True,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,48_10048_North_Yellow_Sea_(2019).json,1,True,True,True,True,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,48_11048_North_Yellow_Sea_(2019).json,1,True,True,True,True,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,48_12048_Southwestern_Yellow_Sea_(2008).json,1,True,True,True,True,True,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [12]:
results_df['PPR2NPP_TE_all'] / results_df['PPR2NPP_1995_TE0.1']

0        6.727376
1       35.675632
2    14719.707168
3      509.680328
4    17406.426488
5       27.111279
6        0.719737
7      471.597899
8     1336.113457
9        1.030785
dtype: float64